## AI coding agent workshop @ MLcon Amsterdam 2026
### presented by Alexey Grigorev

In [1]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url='https://api.groq.com/openai/v1'
)
model = 'openai/gpt-oss-120b'

In [2]:
### system prompt is what the agent is going to do
### user prompt is how an user interacts with agent

In [3]:
system_prompt = "You are a helpful coding assistant. Do not make any assumptions. Look deeply into the query"
user_prompt = "What's in this folder?"

chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model=model,
    input=chat_messages,
)

print(response.output_text)

I can’t see the contents of the folder from here. Could you please provide a directory listing (e.g., the output of `ls -la` on Unix/macOS, `dir` on Windows, or a copy‑paste of the folder tree) or describe the files you see? Once I know what’s inside, I’ll be able to help you with whatever you need—whether it’s understanding the files, writing scripts to process them, or anything else.


In [4]:
import os

def see_file_tree(root_dir: str = ".") -> list[str]:
    """List files and directories in the current project.

    Args:
        root_dir: Directory to list. Defaults to the current directory.
    """
    if not root_dir:
        root_dir = "."
    
    tree = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames[:] = [d for d in dirnames if d not in {".git", ".venv", "__pycache__", ".ipynb_checkpoints","uv.lock"}]

        for name in sorted(dirnames + filenames):
            full_path = os.path.join(dirpath, name)
            tree.append(os.path.relpath(full_path, root_dir))

    return tree

In [5]:
see_file_tree()

['.python-version',
 'README.md',
 'main.py',
 'pyproject.toml',
 'uv.lock',
 'workshop.ipynb']

In [6]:
see_file_tree_description = {
    "type": "function",
    "name": "see_file_tree",
    "description": "List files and directories in the current project.",
    "parameters": {
        "type": "object",
        "properties": {
            "root_dir": {
                "type": "string",
                "description": "Directory to list. Defaults to the current directory.",
                "default": "."
            }
        },
        "required": [],
        "additionalProperties": False,
    },
}

response = openai_client.responses.create(
    model=model,
    input=chat_messages,
    tools=[see_file_tree_description]
)

In [7]:
response.output

[ResponseReasoningItem(id='resp_01kpqpwsg8e58tb1ptpzc2z0px', summary=[], type='reasoning', content=[Content(text='The user asks "What\'s in this folder?" Likely they want to see the current project folder contents. Use see_file_tree function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"root_dir":"."}', call_id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', name='see_file_tree', type='function_call', id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', namespace=None, status='completed')]

In [8]:
call = next(item for item in response.output if item.type == "function_call")

In [9]:
import json
args = json.loads(call.arguments)
args

{'root_dir': '.'}

In [10]:
results = see_file_tree(**args)
results

['.python-version',
 'README.md',
 'main.py',
 'pyproject.toml',
 'uv.lock',
 'workshop.ipynb']

In [11]:
chat_messages

[{'role': 'developer',
  'content': 'You are a helpful coding assistant. Do not make any assumptions. Look deeply into the query'},
 {'role': 'user', 'content': "What's in this folder?"}]

In [12]:
chat_messages.extend(response.output)

In [13]:
chat_messages

[{'role': 'developer',
  'content': 'You are a helpful coding assistant. Do not make any assumptions. Look deeply into the query'},
 {'role': 'user', 'content': "What's in this folder?"},
 ResponseReasoningItem(id='resp_01kpqpwsg8e58tb1ptpzc2z0px', summary=[], type='reasoning', content=[Content(text='The user asks "What\'s in this folder?" Likely they want to see the current project folder contents. Use see_file_tree function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"root_dir":"."}', call_id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', name='see_file_tree', type='function_call', id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', namespace=None, status='completed')]

In [14]:
chat_messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(results),
})

In [15]:
chat_messages

[{'role': 'developer',
  'content': 'You are a helpful coding assistant. Do not make any assumptions. Look deeply into the query'},
 {'role': 'user', 'content': "What's in this folder?"},
 ResponseReasoningItem(id='resp_01kpqpwsg8e58tb1ptpzc2z0px', summary=[], type='reasoning', content=[Content(text='The user asks "What\'s in this folder?" Likely they want to see the current project folder contents. Use see_file_tree function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"root_dir":"."}', call_id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', name='see_file_tree', type='function_call', id='fc_f0167229-7d53-44e7-8873-ae134cc923b5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_f0167229-7d53-44e7-8873-ae134cc923b5',
  'output': '[".python-version", "README.md", "main.py", "pyproject.toml", "uv.lock", "workshop.ipynb"]'}]

In [18]:
response = openai_client.responses.create(
    model=model,
    input=chat_messages,
    tools=[see_file_tree_description]
)

In [19]:
print(response.output_text)

Here’s what’s in the current project folder:

- **`.python-version`** – Specifies the Python version for the project (often used by tools like pyenv).  
- **`README.md`** – Project documentation and overview.  
- **`main.py`** – The main Python script for the application.  
- **`pyproject.toml`** – Configuration file for build tools, dependencies, and package metadata (PEP 518).  
- **`uv.lock`** – Lock file generated by the **uv** package manager, pinning exact dependency versions.  
- **`workshop.ipynb`** – A Jupyter notebook, likely containing examples, tutorials, or interactive code for the workshop.


In [44]:
def read_file(filepath: str) -> str:
    """Read the contents of a file in the current project.

    Args:
        filepath: Path to the file to read.
    """
    try:
        if filepath == 'uv.lock':
            return "File ignored: uv.lock files are not readable."
        
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: file '{filepath}' not found."

read_file_description = {
    "type": "function",
    "name": "read_file",
    "description": "Read the contents of a file in the current project.",
    "parameters": {
        "type": "object",
        "properties": {
            "filepath": {
                "type": "string",
                "description": "Path to the file to read.",
            }
        },
        "required": ["filepath"],
        "additionalProperties": False,
    },
}

user_prompt = "What dependencies does this project have?"

chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [45]:
response = openai_client.responses.create(
    model=model,
    input=chat_messages,
    tools=[see_file_tree_description, read_file_description]
)

In [46]:
response.output

[ResponseReasoningItem(id='resp_01kpqqj0rgex6tn7e0nfqbq38e', summary=[], type='reasoning', content=[Content(text="We need to list dependencies. Likely from package.json. Let's inspect project files.", type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"root_dir":"."}', call_id='fc_1dd124a9-a946-4603-aaaa-82eddb3970fd', name='see_file_tree', type='function_call', id='fc_1dd124a9-a946-4603-aaaa-82eddb3970fd', namespace=None, status='completed')]

In [34]:
item = response.output[0]
item.content[0].text

"We need to answer what dependencies this project has. The pyproject.toml lists dependencies: jupyter>=1.1.1, openai>=2.32.0, toyaikit>=0.0.9. Also maybe other files like uv.lock may list dependencies but not needed. We'll answer accordingly."

In [35]:
chat_messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        f_name = item.name
        args = json.loads(item.arguments)
        print(f'tool_call: {f_name}({args})')

        if f_name == 'see_file_tree':
            result = see_file_tree(**args)
        elif f_name == 'read_file':
            result = read_file(**args)
        else:
            raise Exception(f'unknown function {f_name}')

        chat_messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result),
        })

In [36]:
chat_messages

[{'role': 'developer',
  'content': 'You are a helpful coding assistant. Do not make any assumptions. Look deeply into the query'},
 {'role': 'user', 'content': 'What dependencies does this project have?'},
 ResponseReasoningItem(id='resp_01kpqpxt47ermr5v681g8jafsv', summary=[], type='reasoning', content=[Content(text='We need to find dependencies. Likely a package.json file. Use see_file_tree.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"root_dir":"."}', call_id='fc_4a546dfa-0a58-455a-8d3a-85a15b14c779', name='see_file_tree', type='function_call', id='fc_4a546dfa-0a58-455a-8d3a-85a15b14c779', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_4a546dfa-0a58-455a-8d3a-85a15b14c779',
  'output': '[".python-version", "README.md", "main.py", "pyproject.toml", "uv.lock", "workshop.ipynb"]'},
 ResponseReasoningItem(id='resp_01kpqpyekaepsb9z856cf56801', summary=[], type='reasoning', content=

In [47]:
chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

while True:
    response = openai_client.responses.create(
        model=model,
        input=chat_messages,
        tools=[see_file_tree_description, read_file_description]
    )

    has_tool_calls = False

    chat_messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            has_tool_calls = True

            f_name = item.name
            args = json.loads(item.arguments)
            print(f'tool_call: {f_name}({args})')

            if f_name == 'see_file_tree':
                result = see_file_tree(**args)
            elif f_name == 'read_file':
                result = read_file(**args)
            else:
                raise Exception(f'unknown function {f_name}')

            chat_messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result),
            })
        elif item.type == 'message':
            print(item.content[0].text)

    if not has_tool_calls:
        break

tool_call: see_file_tree({'root_dir': '.'})
tool_call: read_file({'filepath': 'pyproject.toml'})
The project’s dependencies are declared in **`pyproject.toml`** under the `[project]` table:

| Dependency | Minimum version |
|------------|-----------------|
| `jupyter`  | `>=1.1.1` |
| `openai`   | `>=2.32.0` |
| `toyaikit` | `>=0.0.9` |

Additionally, the project requires Python **3.12 or newer** (`requires-python = ">=3.12"`). No other dependency files (e.g., `requirements.txt`) are present.


In [47]:
!uv add toyaikit

Resolved 114 packages in 1.08s                                       
Prepared 4 packages in 95ms                                              
░░░░░░░░░░░░░░░░░░░░ [0/4] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 4 packages in 115ms                               
 + anthropic==0.96.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.56
 + toyaikit==0.0.9


In [37]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

In [38]:
from decimal import Decimal
from toyaikit.pricing import FALLBACK_PRICING

if model == "llama-3.3-70b-versatile":
    FALLBACK_PRICING[model] = {
        "input": Decimal("0.59"),
        "output": Decimal("0.79"),
    }

In [48]:
tools_obj = Tools()
tools_obj.add_tool(see_file_tree, see_file_tree_description)
tools_obj.add_tool(read_file, read_file_description)

In [49]:
chat_interface = IPythonChatInterface()
llm_client = OpenAIClient(client=openai_client, model=model)

runner = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=system_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [50]:
results = runner.run()

You: what are the dependencies for this project


-> Response received


-> Response received


-> Response received


Dependency,Version constraint
jupyter,>=1.1.1
openai,>=2.32.0
toyaikit,>=0.0.9


LookupError: ('Please check model name. Use list_all_models function to see list of supported models.', LookupError("Unable to find provider with model matching 'openai/gpt-oss-120b'"))

In [53]:
llm_client = OpenAIClient(client=openai_client, model=model)
available_models = llm_client.list_all_models()
print(available_models)

AttributeError: 'OpenAIClient' object has no attribute 'list_all_models'

In [54]:
models = openai_client.models.list()
for model in models.data:
    print(model.id)

openai/gpt-oss-120b
groq/compound-mini
canopylabs/orpheus-arabic-saudi
meta-llama/llama-4-scout-17b-16e-instruct
qwen/qwen3-32b
whisper-large-v3
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
allam-2-7b
llama-3.3-70b-versatile
openai/gpt-oss-20b
groq/compound
llama-3.1-8b-instant
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m


In [55]:
import os
from pathlib import Path

class ProjectTools:
    def __init__(self, project_dir: Path):
        self.project_dir = project_dir

    def see_file_tree(self, root_dir: str = ".") -> list[str]:
        """List files and directories in the project.

        Args:
            root_dir: Directory to list relative to the project root.
        """
        abs_root = self.project_dir / root_dir
        tree = []

        for dirpath, dirnames, filenames in os.walk(abs_root):
            dirnames[:] = [d for d in dirnames if d not in {".git", ".venv", "__pycache__"}]

            for name in sorted(dirnames + filenames):
                full_path = os.path.join(dirpath, name)
                tree.append(os.path.relpath(full_path, self.project_dir))

        return tree

    def read_file(self, filepath: str) -> str:
        """Read the contents of a file relative to the project root.

        Args:
            filepath: Path to the file to read.
        """
        abs_path = self.project_dir / filepath
        try:
            with open(abs_path, "r", encoding="utf-8") as f:
                return f.read()
        except FileNotFoundError:
            return f"Error: file '{filepath}' not found."

    def search_in_files(self, search_term: str) -> list[str]:
        """Search for a string in project files.

        Args:
            search_term: Text to search for.
        """
        matches = []

        for dirpath, dirnames, filenames in os.walk(self.project_dir):
            dirnames[:] = [d for d in dirnames if d not in {".git", ".venv", "__pycache__"}]

            for filename in filenames:
                abs_path = Path(dirpath) / filename

                try:
                    content = abs_path.read_text(encoding="utf-8")
                except Exception:
                    continue

                if search_term in content:
                    matches.append(str(abs_path.relative_to(self.project_dir)))

        return matches

In [59]:
from pathlib import Path 
project_tools = ProjectTools(Path('.'))

In [62]:
project_tools.see_file_tree()

['.ipynb_checkpoints',
 '.python-version',
 'README.md',
 'main.py',
 'pyproject.toml',
 'uv.lock',
 'workshop.ipynb',
 '.ipynb_checkpoints/workshop-checkpoint.ipynb']

In [65]:
tools_obj = Tools()
tools_obj.add_tools(project_tools)

In [75]:
import tools
agent_tools = tools.AgentTools(Path("."))

tools_obj = Tools()
tools_obj.add_tools(agent_tools)

In [67]:
project_tools.search_in_files('agent')

['tools.py',
 'main.py',
 'workshop.ipynb',
 'uv.lock',
 'README.md',
 'pyproject.toml',
 '.ipynb_checkpoints/tools-checkpoint.py']

In [76]:
if model == "llama-3.3-70b-versatile":
    FALLBACK_PRICING[model] = {
        "input": Decimal("0.59"),
        "output": Decimal("0.79"),
    }

chat_interface = IPythonChatInterface()
llm_client = OpenAIClient(client=openai_client, model="llama-3.3-70b-versatile")

runner = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=system_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [77]:
runner.run();

You: create a browser based vanilla js snake game in snake.html


-> Response received


-> Response received


LookupError: ('Please check model name. Use list_all_models function to see list of supported models.', LookupError("Unable to find provider with model matching 'llama-3.3-70b-versatile'"))